# Introduction to HLT Project (Template)

- Student(s) Name(s): Anton Teerioja
- Date: 17.5.2026
- Chosen Corpus: IMDb
- Contributions (if group project): None

### Corpus information

- Description of the chosen corpus: IMDb
- Paper(s) and other published materials related to the corpus: 
- Original paper using this dataset: https://aclanthology.org/P11-1015.pdf
- Current best model using this dataset: https://arxiv.org/pdf/1906.08237 

- State-of-the-art performance (best published results) on this corpus: https://nlpprogress.com/english/sentiment_analysis.html
- The top 3 models for the IMDb dataset are currently:
1. XLNet:           96.21%        
XLNet Generalized Autoregressive Pretraining for Language Understanding https://arxiv.org/pdf/1906.08237.pdf 
2. BERT_large+ITPT: 95.79%        
How to Fine-Tune BERT for Text Classification? https://arxiv.org/pdf/1905.05583.pdf
3. BERT_base+ITPT:  95.63%        
How to Fine-Tune BERT for Text Classification? https://arxiv.org/pdf/1905.05583.pdf

---

## 1. Setup

In [2]:
# Your code to install and import libraries etc. here
import datasets
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix



---

## 2. Data download and preprocessing

### 2.1. Download the corpus

In [3]:
# Your code to download the corpus here
dset = datasets.load_dataset("imdb")

### 2.2. Preprocessing

In [4]:
# Your code for any necessary preprocessing here
train = dset["train"]
test = dset["test"]

#Split train into train and validation with stratification for class balance
train_validation = train.train_test_split(test_size=0.1, seed=420, stratify_by_column='label')

train = train_validation["train"]
validation = train_validation["test"]

X_train = train["text"]
X_val = validation["text"]
X_test = test["text"]

y_train = train["label"]
y_val = validation["label"]
y_test = test["label"]

In [5]:
#TF-IDF Vectorization
tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.95,
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf   = tfidf.transform(X_val)
X_test_tfidf  = tfidf.transform(X_test)

print(f"TF-IDF feature matrix shape: {X_train_tfidf.shape}")

TF-IDF feature matrix shape: (22500, 20000)


---

## 3. Machine learning model

### 3.1. Model training

In [6]:
# Your code to train the machine learning model on the training set and evaluate the performance on the validation set here
mlp_baseline = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation="relu",
    solver="adam",
    max_iter=30,
    batch_size=256,
    learning_rate_init=0.001,
    early_stopping=True,
    validation_fraction=0.1,
    random_state=42,
    verbose=True
)

mlp_baseline.fit(X_train_tfidf, y_train)

Iteration 1, loss = 0.42570068
Validation score: 0.894667
Iteration 2, loss = 0.13435837
Validation score: 0.892000
Iteration 3, loss = 0.04915424
Validation score: 0.886667
Iteration 4, loss = 0.01679048
Validation score: 0.884444
Iteration 5, loss = 0.00688683
Validation score: 0.885778
Iteration 6, loss = 0.00341433
Validation score: 0.885333
Iteration 7, loss = 0.00192752
Validation score: 0.888444
Iteration 8, loss = 0.00108359
Validation score: 0.886222
Iteration 9, loss = 0.00081495
Validation score: 0.887111
Iteration 10, loss = 0.00069709
Validation score: 0.886667
Iteration 11, loss = 0.00063208
Validation score: 0.886667
Iteration 12, loss = 0.00059223
Validation score: 0.887111
Validation score did not improve more than tol=0.000100 for 10 consecutive epochs. Stopping.


,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(128, ...)"
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'relu'
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'adam'
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.For an example usage and visualization of varying regularization, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_alpha.py`.",0.0001
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the classifier will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",256
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.Only used when ``solver='sgd'``.",'constant'
,"learning_rate_init learning_rate_init: float, default=0.001The initial learning rate used. It controls the step-sizein updating the weights. Only used when solver='sgd' or 'adam'.",0.001
,"power_t power_t: float, default=0.5The exponent for inverse scaling learning rate.It is used in updating effective learning rate when the learning_rateis set to 'invscaling'. Only used when solver='sgd'.",0.5
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",30
,"shuffle shuffle: bool, default=TrueWhether to shuffle samples in each iteration. Only used whensolver='sgd' or 'adam'.",True
,"random_state random_state: int, RandomState instance, default=NoneDetermines random number generation for weights and biasinitialization, train-test split if early stopping is used, and batchsampling when solver='sgd' or 'adam'.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",42


In [7]:
#Evaluate the baseline performance on the validation set
y_val_pred = mlp_baseline.predict(X_val_tfidf)

print("Validation Results")
print(f"Accuracy: {accuracy_score(y_val, y_val_pred):.4f}\n")
print(classification_report(y_val, y_val_pred, target_names=["Negative", "Positive"]))
print("Confusion Matrix:")
print(confusion_matrix(y_val, y_val_pred))

Validation Results
Accuracy: 0.9104

              precision    recall  f1-score   support

    Negative       0.90      0.93      0.91      1250
    Positive       0.92      0.90      0.91      1250

    accuracy                           0.91      2500
   macro avg       0.91      0.91      0.91      2500
weighted avg       0.91      0.91      0.91      2500

Confusion Matrix:
[[1157   93]
 [ 131 1119]]


### 3.2 Hyperparameter optimization

In [8]:
# Your code for hyperparameter optimization here
param_grid = {
    "hidden_layer_sizes": [(128,), (256, 128), (128, 64, 32)],
    "activation": ["relu", "tanh"],
    "learning_rate_init": [0.001, 0.0005],
    "alpha": [0.0001, 0.001],
    "batch_size": [256, 512]
}

grid_search = GridSearchCV(
    estimator=MLPClassifier(
        solver="adam",
        max_iter=30,
        early_stopping=True,
        validation_fraction=0.1,
        random_state=42
    ),
    param_grid=param_grid,
    scoring="accuracy",
    cv=3,
    n_jobs=-1,
    verbose=10
)

#To avoid another 82 min search save the results
#grid_search.fit(X_train_tfidf, y_train)
grid_search = joblib.load('grid_search_results.pkl')

print("\nBest parameters found:")
for k, v in grid_search.best_params_.items():
    print(f"  {k}: {v}")
print(f"\nBest CV accuracy: {grid_search.best_score_:.4f}")


Best parameters found:
  activation: relu
  alpha: 0.001
  batch_size: 256
  hidden_layer_sizes: (256, 128)
  learning_rate_init: 0.0005

Best CV accuracy: 0.9010


In [9]:
#Helper line to save the result
#joblib.dump(grid_search, 'grid_search_results.pkl')

In [10]:
best_mlp = grid_search.best_estimator_

y_val_pred_best = best_mlp.predict(X_val_tfidf)

print("Optimized Validation Results\n")
print(f"Accuracy: {accuracy_score(y_val, y_val_pred_best):.4f}\n")
print(classification_report(y_val, y_val_pred_best, target_names=["Negative", "Positive"]))
print("Confusion Matrix:")
print(confusion_matrix(y_val, y_val_pred_best))

Optimized Validation Results

Accuracy: 0.9052

              precision    recall  f1-score   support

    Negative       0.92      0.89      0.90      1250
    Positive       0.89      0.92      0.91      1250

    accuracy                           0.91      2500
   macro avg       0.91      0.91      0.91      2500
weighted avg       0.91      0.91      0.91      2500

Confusion Matrix:
[[1113  137]
 [ 100 1150]]


In [11]:
#Retrain on best parameters
final_mlp = MLPClassifier(
    **grid_search.best_params_,
    solver="adam",
    max_iter=50,
    early_stopping=True,
    validation_fraction=0.1,
    random_state=42
)

#Load the best model
#final_mlp.fit(X_train_tfidf, y_train)
final_mlp = joblib.load("final_mlp.pkl")

In [12]:
#Helper to save the best model
#joblib.dump(final_mlp, "final_mlp.pkl")

### 3.3. Evaluation on test set

In [13]:
# Your code to evaluate the final model on the test set here
y_test_pred = final_mlp.predict(X_test_tfidf)

print("Test Set Results\n")
print(f"Accuracy: {accuracy_score(y_test, y_test_pred):.4f}\n")
print(classification_report(y_test, y_test_pred, target_names=["Negative", "Positive"]))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

Test Set Results

Accuracy: 0.8972

              precision    recall  f1-score   support

    Negative       0.90      0.89      0.90     12500
    Positive       0.89      0.90      0.90     12500

    accuracy                           0.90     25000
   macro avg       0.90      0.90      0.90     25000
weighted avg       0.90      0.90      0.90     25000

Confusion Matrix:
[[11163  1337]
 [ 1234 11266]]


In [14]:
print(f"{'Model':<14} | {'Val Acc':>10} | {'Test Acc':>10} |")
print(f"{'Baseline MLP':<14} | {accuracy_score(y_val, y_val_pred):>10.4f} | {'—':>10} |")
print(f"{'Optimized MLP':<14} | {accuracy_score(y_val, y_val_pred_best):>10.4f} | {accuracy_score(y_test, y_test_pred):>10.4f} |")

Model          |    Val Acc |   Test Acc |
Baseline MLP   |     0.9104 |          — |
Optimized MLP  |     0.9052 |     0.8972 |


---

## 4. Results and summary

### 4.1 Corpus insights

The corpus consists of 50000 labelled movie reviews from IMDB and another 50000 unlabelled reviews. The limit for reviews per movie was set to 30, because some movies have less reviews. The reviews are only labelled when there is a strong positive or negative score (>= 7/10 and <= 4/10). This corpus is one of the most popular benchmarks for standard text-classification. The reviews are balanced between positive and negative reviews. 

The dataset is evenly divided into training and test sets. 

### 4.2 Results

First the text data was vectorized using TF-IDF Vectorization. This was done on all of the training, validation and test data. This vectorized data was then used to train the first MLP-model using hyperparameters that were sensible for a blind attempt. 

The result of the first MLP was impressive as the model achieved an accuracy of 91% on the validation data. This is already better than the first solution introduced in the first paper.

Then using hyperparameter optimization with an array of parameters the best result wasn't that much better than the original.

The resulting model had a validation accuracy of 91%, so there wasn't much improvement using these hyperparameters. This could be improved by searching through more combinations of hyperparameters, but this already took 81 mins to search through these so I won't be looking into that now.

In the end the best model was used on the test data and the final result was an accuracy of 89.72% Which is slightly better than the original paper.

### 4.3 Relation to state of the art

The state-of-the-art performance on this dataset is currently the XLNet. This solution is offered as an improvement ofer BERT. It outperforms BERT on natural language inference and scores higher on the IMDb dataset.

The Accuracy score of XLNet is 96.21% which is much higher than the 89.72% that I got with my training.

---

## 5. Bonus Task (optional)

### 5.1. Annotating out-of-domain documents

(Briefly describe the chosen out-of-domain documents)

(Briefly describe the process of annotation)

### 5.2 Conversion into dataset

In [15]:
# Your code to convert the annotations into a dataset here

### 5.3. Model evaluation on out-of-domain test set

In [16]:
# Your code to evaluate the model on the out-of-domain test set here

### 5.4 Bonus task results

(Present the results of the evaluation on the out-of-domain test set)

### 5.5. Annotated data

In [17]:
# Include your annotated out-of-domain data here